# Day 5 — Sanchari YouTube Analytics: Bronze → Silver

**Goal:** Create a typed, deduplicated, auditable Silver Delta table from existing Bronze records without rerunning ingestion or altering Bronze.

**Source:** `workspace.sanchari_bronze.youtube_video_daily_raw` (37,327 rows at Day 4 completion).  
**Target:** `workspace.sanchari_silver.youtube_video_daily`  
**Rejected rows:** `workspace.sanchari_silver.youtube_video_daily_quarantine`

Read and execute each numbered cell in order. This is a **batch reconciliation with an idempotent MERGE**, not yet a high-volume incremental-read implementation. For 37k rows the complete Bronze rescan is deliberate; production optimization can follow.

**Business key:** `(channel_id, video_id, reporting_date)` — one current record per channel, video and analytics day. The most recent extraction wins even if the metrics decreased. Original Bronze history remains intact.

## 1. Configuration and Bronze checks

We do not use `DROP`, `TRUNCATE`, `DELETE` or `OVERWRITE` on Bronze. Confirm the existing table schema and the source anomaly before building Silver.

In [ ]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, LongType
from delta.tables import DeltaTable

BRONZE_TABLE = "workspace.sanchari_bronze.youtube_video_daily_raw"
SILVER_TABLE = "workspace.sanchari_silver.youtube_video_daily"
QUARANTINE_TABLE = "workspace.sanchari_silver.youtube_video_daily_quarantine"

# Confirmed ownership of the Sanchari Travel Shorts YouTube channel.
VERIFIED_CHANNEL_ID = "UCGpXEis-qat1YG04joiG_Vw"
# From the Day 4 JSON example. Compare the EXACT case-sensitive ID below
# with the output of the diagnostic query. Capital I and lowercase l differ.
VERIFIED_MISSING_VIDEO_ID = "3_MPPElISLA"

bronze = spark.table(BRONZE_TABLE)
bronze_count_before = bronze.count()
print("Bronze rows (unchanged source):", bronze_count_before)
bronze.printSchema()

missing_channel = bronze.where(
    F.col("channel_id").isNull() | (F.trim(F.col("channel_id")) == "")
)
print("Missing channel ID rows:", missing_channel.count())
display(
    missing_channel.groupBy("video_id")
    .agg(F.count("*").alias("rows"), F.min("reporting_date").alias("first_date"),
         F.max("reporting_date").alias("last_date"))
    .orderBy(F.desc("rows"))
)
# Use repr() to make letter-case differences easier to spot.
print("Missing video IDs:", [repr(x.video_id) for x in missing_channel.select("video_id").distinct().collect()])

## 2. Parse the raw JSON and normalize types

`raw_json` preserves the source record. We project its nested `metrics` into typed columns, parse its source extraction timestamp and standardize the video/date keys. `estimatedMinutesWatched` is **minutes**; `averageViewDuration` is **seconds**.

In [ ]:
metrics_schema = StructType([
    StructField("views", LongType(), True),
    StructField("estimatedMinutesWatched", LongType(), True),
    StructField("averageViewDuration", LongType(), True),
    StructField("likes", LongType(), True),
    StructField("comments", LongType(), True),
    StructField("shares", LongType(), True),
])

json_schema = StructType([
    StructField("extract_run_id", StringType(), True),
    StructField("ingested_at_utc", StringType(), True),
    StructField("source_endpoint", StringType(), True),
    StructField("video_id", StringType(), True),
    StructField("reporting_date", StringType(), True),
    StructField("metrics", metrics_schema, True),
])

parsed = (
    bronze
    .withColumn("j", F.from_json(F.col("raw_json"), json_schema))
    .withColumn("bronze_video_id", F.when(F.length(F.trim("video_id")) > 0, F.trim("video_id")))
    .withColumn("json_video_id", F.when(F.length(F.trim(F.col("j.video_id"))) > 0, F.trim(F.col("j.video_id"))))
    .withColumn("video_id_clean", F.coalesce("bronze_video_id", "json_video_id"))
    .withColumn("bronze_date", F.expr("try_cast(reporting_date AS DATE)"))
    .withColumn("json_date", F.expr("try_cast(j.reporting_date AS DATE)"))
    .withColumn("reporting_date_clean", F.coalesce("bronze_date", "json_date"))
    .withColumn("channel_id_from_bronze", F.when(F.length(F.trim("channel_id")) > 0, F.trim("channel_id")))
    .withColumn("source_extracted_at", F.expr("try_cast(j.ingested_at_utc AS TIMESTAMP)"))
    .withColumn("record_freshness_at", F.coalesce(
        F.col("source_extracted_at"), F.col("bronze_ingested_at").cast("timestamp"),
        F.col("source_file_modified_at").cast("timestamp")
    ))
    .withColumn("record_sha256_clean", F.coalesce(F.col("record_sha256"), F.sha2("raw_json", 256)))
    .withColumn("views", F.col("j.metrics.views"))
    .withColumn("estimated_minutes_watched", F.col("j.metrics.estimatedMinutesWatched"))
    .withColumn("average_view_duration_seconds", F.col("j.metrics.averageViewDuration"))
    .withColumn("likes", F.col("j.metrics.likes"))
    .withColumn("comments", F.col("j.metrics.comments"))
    .withColumn("shares", F.col("j.metrics.shares"))
)

parsed.select("video_id_clean", "reporting_date_clean", "views", 
              "estimated_minutes_watched", "average_view_duration_seconds", 
              "source_extracted_at").show(5, truncate=False)

## 3. Resolve channel IDs with evidence

First, obtain a channel ID from other Bronze rows for the *same video* only when the mapping is unambiguous. If no existing mapping exists, apply the **single video-specific ownership confirmation** from Day 4. Never fill all missing channel IDs with a blanket default: future files could include other channels.

If the six null rows are not resolved, inspect the case-sensitive `VERIFIED_MISSING_VIDEO_ID` string in Cell 1 before proceeding.

In [ ]:
known_video_channel = (
    parsed.where(F.col("channel_id_from_bronze").isNotNull())
    .groupBy("video_id_clean")
    .agg(F.countDistinct("channel_id_from_bronze").alias("known_channel_id_count"),
         F.max("channel_id_from_bronze").alias("mapped_channel_id"))
)

staged = (
    parsed.join(known_video_channel, on="video_id_clean", how="left")
    .withColumn("channel_id_resolved", F.coalesce(
        F.col("channel_id_from_bronze"),
        F.when(F.col("known_channel_id_count") == 1, F.col("mapped_channel_id")),
        F.when(F.col("video_id_clean") == F.lit(VERIFIED_MISSING_VIDEO_ID),
               F.lit(VERIFIED_CHANNEL_ID))
    ))
    .withColumn("channel_id_resolution", 
        F.when(F.col("channel_id_from_bronze").isNotNull(), F.lit("source"))
         .when(F.col("known_channel_id_count") == 1, F.lit("same_video_mapping"))
         .when(F.col("video_id_clean") == F.lit(VERIFIED_MISSING_VIDEO_ID),
               F.lit("manual_verified_video"))
         .otherwise(F.lit("unresolved")))
)

print("Resolution of rows with missing original channel IDs:")
display(staged.where(F.col("channel_id_from_bronze").isNull())
        .groupBy("channel_id_resolution", "channel_id_resolved").count())

## 4. Data-quality rules and quarantine

A rejected record must remain visible for investigation; it is not silently dropped. The quarantine is a derived **Silver** table that may safely be replaced on a full reconciliation. Bronze is never rewritten.

We reject malformed JSON, missing identity fields, unexpected channel ownership, conflicting IDs/dates, missing metrics, negative metrics and records with no usable extraction/ingestion timestamp. Zero metrics are allowed.

In [ ]:
from pyspark.sql import functions as F

metrics = [
    "views",
    "estimated_minutes_watched",
    "average_view_duration_seconds",
    "likes",
    "comments",
    "shares"
]

nonnegative_metrics = [m for m in metrics if m != "likes"]

missing_metric = F.lit(False)
negative_metric = F.lit(False)

for m in metrics:
    missing_metric = missing_metric | F.col(m).isNull()

for m in nonnegative_metrics:
    negative_metric = negative_metric | (F.col(m) < 0)
    

checked = staged.withColumn("rejection_reason", F.concat_ws("; ",
    F.when(F.col("j").isNull(), F.lit("unparseable_json")),
    F.when(F.col("video_id_clean").isNull(), F.lit("missing_video_id")),
    F.when(F.col("reporting_date_clean").isNull(), F.lit("invalid_reporting_date")),
    F.when(F.col("channel_id_resolved").isNull(), F.lit("unresolved_channel_id")),
    F.when(F.col("channel_id_resolved").isNotNull() &
           (F.col("channel_id_resolved") != F.lit(VERIFIED_CHANNEL_ID)),
           F.lit("unexpected_channel_id")),
    F.when(F.col("j.source_endpoint").isNotNull() &
           (F.col("j.source_endpoint") != F.lit("youtubeAnalytics.reports.query")),
           F.lit("unexpected_source_endpoint")),
    F.when(F.col("known_channel_id_count") > 1, F.lit("conflicting_channel_mapping")),
    F.when(F.col("bronze_video_id").isNotNull() & F.col("json_video_id").isNotNull() &
           (F.col("bronze_video_id") != F.col("json_video_id")), F.lit("video_id_conflict")),
    F.when(F.col("bronze_date").isNotNull() & F.col("json_date").isNotNull() &
           (F.col("bronze_date") != F.col("json_date")), F.lit("reporting_date_conflict")),
    F.when(missing_metric, F.lit("missing_or_invalid_metric")),
    F.when(negative_metric, F.lit("negative_metric")),
    F.when(F.col("record_freshness_at").isNull(), F.lit("missing_source_timestamp"))
))

valid = checked.where(F.col("rejection_reason") == "")
rejected = checked.where(F.col("rejection_reason") != "")

quarantine = rejected.select(
    F.col("video_id_clean").alias("video_id"),
    F.col("reporting_date_clean").alias("reporting_date"),
    F.col("channel_id_from_bronze").alias("original_channel_id"),
    F.col("channel_id_resolved").alias("resolved_channel_id"),
    "rejection_reason", "run_id", "record_sha256", "source_file_path", "raw_json",
    "bronze_ingested_at"
)
quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(QUARANTINE_TABLE)

print("Valid Bronze records:", valid.count())
print("Quarantined records:", rejected.count())
display(checked.groupBy("rejection_reason").count().orderBy(F.desc("count")))
print("Original missing channel IDs still unresolved:",
      checked.where(F.col("channel_id_from_bronze").isNull() &
                    F.col("channel_id_resolved").isNull()).count())

## 5. Deduplicate overlapping extraction runs

For each `(channel, video, reporting_date)` choose the row with the newest **source extraction** timestamp. The Bronze ingestion timestamp is only a fallback: an older backfill might have been uploaded later than a newer extraction. SHA-256 and file path give deterministic tie-breakers. Retain source lineage in Silver.

In [ ]:
key_cols = ["channel_id_resolved", "video_id_clean", "reporting_date_clean"]
window = Window.partitionBy(*key_cols).orderBy(
    F.col("record_freshness_at").desc_nulls_last(),
    F.col("record_sha256_clean").desc_nulls_last(),
    F.col("source_file_path").desc_nulls_last()
)

winners = (
    valid.withColumn("row_rank", F.row_number().over(window))
    .where(F.col("row_rank") == 1)
    .select(
        F.col("channel_id_resolved").alias("channel_id"),
        F.col("video_id_clean").alias("video_id"),
        F.col("reporting_date_clean").alias("reporting_date"),
        *[F.col(x) for x in metrics],
        "source_extracted_at", "record_freshness_at",
        F.col("run_id").alias("source_run_id"), "source_file_path",
        F.col("record_sha256_clean").alias("record_sha256"),
        "channel_id_resolution", F.current_timestamp().alias("silver_processed_at")
    )
)

n_valid = valid.count()
n_winners = winners.count()
print("Valid source rows:", n_valid)
print("Unique video-day winners:", n_winners)
print("Overlapping records reconciled:", n_valid - n_winners)
assert winners.select("channel_id", "video_id", "reporting_date").distinct().count() == n_winners
winners.orderBy(F.desc("reporting_date"), "video_id").show(10, truncate=False)

## 6. Idempotent Delta MERGE (insert new keys, update newer versions)

Use **MERGE**, not blind append or overwrite. A repeated run leaves previously selected records unchanged. The source is deduplicated **before** MERGE, avoiding ambiguous matches.

This cell can be rerun safely. It rescans all Bronze rows on each run for simplicity, but only writes actual inserts/updates.

In [ ]:
if not spark.catalog.tableExists(SILVER_TABLE):
    winners.limit(0).write.format("delta").saveAsTable(SILVER_TABLE)

silver_delta = DeltaTable.forName(spark, SILVER_TABLE)
(
    silver_delta.alias("t")
    .merge(winners.alias("s"),
           "t.channel_id = s.channel_id AND t.video_id = s.video_id "
           "AND t.reporting_date = s.reporting_date")
    .whenMatchedUpdateAll(condition=(
        "s.record_freshness_at > t.record_freshness_at OR "
        "(s.record_freshness_at = t.record_freshness_at AND "
        "s.record_sha256 > t.record_sha256)"
    ))
    .whenNotMatchedInsertAll()
    .execute()
)
print("Silver MERGE completed:", SILVER_TABLE)

## 7. Validate output and protect Bronze

These checks are intended as **executable acceptance tests**. Do not proceed to Gold until they pass. `Silver` has fewer rows than Bronze because extraction runs overlap; that is expected. Its exact row count is discovered, not guessed.

In [ ]:
silver = spark.table(SILVER_TABLE)
n_silver = silver.count()
expected = valid.select(*key_cols).distinct().count()

duplicate_groups = (silver.groupBy("channel_id", "video_id", "reporting_date")
                    .count().where(F.col("count") > 1).count())
null_key_count = silver.where(
    F.col("channel_id").isNull() | F.col("video_id").isNull() |
    F.col("reporting_date").isNull()
).count()
negative_count = silver.where(
    (F.col("views") < 0) |
    (F.col("estimated_minutes_watched") < 0) |
    (F.col("average_view_duration_seconds") < 0) |
    (F.col("comments") < 0) |
    (F.col("shares") < 0)
).count()

negative_likes_count = silver.where(
    F.col("likes") < 0
).count()

print("Valid negative-like observations:", negative_likes_count)
bronze_count_after = spark.table(BRONZE_TABLE).count()

print(f"Bronze before / after: {bronze_count_before:,} / {bronze_count_after:,}")
print(f"Silver actual / expected: {n_silver:,} / {expected:,}")
print(f"Duplicate key groups: {duplicate_groups}")
print(f"Null key rows: {null_key_count}")
print(f"Negative metric rows: {negative_count}")
print(f"Quarantined rows: {spark.table(QUARANTINE_TABLE).count()}")

assert bronze_count_after == bronze_count_before, "Bronze count unexpectedly changed"
assert n_silver == expected, "Silver row count differs from unique valid business keys"
assert duplicate_groups == 0, "Silver contains duplicate business keys"
assert null_key_count == 0, "Silver contains null business keys"
assert negative_count == 0, "Silver contains negative metrics"
print("PASS: Day 5 Silver acceptance tests")
display(silver.orderBy(F.desc("reporting_date"), "video_id").limit(10))

## 8. Delta history and optional idempotence test

Rerun the **Delta MERGE code cell in Section 6**, then run the inspection below. Matching keys that have the same source timestamp and SHA-256 are not updated. Inspect MERGE operation metrics to confirm zero inserts and updates when the underlying Bronze data is unchanged. This result must be verified; the first successful MERGE alone does not prove idempotence.

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}")
        .select("version", "timestamp", "operation", "operationMetrics")
        .limit(5))

display(spark.sql(f"""
SELECT channel_id, channel_id_resolution, COUNT(*) AS silver_rows
FROM {SILVER_TABLE}
GROUP BY channel_id, channel_id_resolution
ORDER BY silver_rows DESC
"""))

## 9. Engineering notes, trade-offs, and limitations

- **Bronze preservation:** The raw ingestion layer is not modified; it retains source JSON and lineage for replay and troubleshooting.
- **Business grain:** Silver keeps one record per `(channel_id, video_id, reporting_date)`; overlapping extracts are resolved by source freshness and deterministic tie-breakers.
- **Data quality:** Unusable records are written to a quarantine table. A known missing channel ID is resolved only for a specifically verified video, not via a global default.
- **Updates:** A Delta `MERGE` inserts new keys and updates existing keys only when a newer source record is available. Past YouTube metric values can be revised downward.
- **Metrics:** Daily likes may be negative; average view duration is not additive and must not be directly summed in Gold.

### Scope and follow-up checks

This is a **full-Bronze rescan and reconciliation** suited to the current dataset size, not a true incremental source read or a scheduled production pipeline. Before claiming rerun idempotence, run the MERGE twice without changing Bronze and check Delta history for zero inserts/updates on the second run. For scaling, evaluate ingestion checkpoints or Change Data Feed, batch-specific quality metrics, exception handling and operational monitoring.

Day 5 validation output showed 37,327 Bronze rows, 36,251 unique Silver video-day rows, zero quarantined rows and zero duplicate key groups. These results refer to the recorded execution, not a live guarantee for future runs.

**Suggested repo path:** `databricks/02_silver_youtube_daily.ipynb`. Source outputs have been cleared for publication; re-run in Databricks to reproduce the checks.


In [ ]:
from pyspark.sql import functions as F

metric_columns = [
    "views",
    "estimated_minutes_watched",
    "average_view_duration_seconds",
    "likes",
    "comments",
    "shares"
]

# Identify which metrics are negative
display(
    rejected.agg(*[
        F.sum(
            F.when(F.col(m) < 0, 1).otherwise(0)
        ).alias(m)
        for m in metric_columns
    ])
)

# Inspect sample rejected observations
display(
    rejected.select(
        "video_id_clean",
        "reporting_date_clean",
        *metric_columns,
        "rejection_reason",
        "run_id"
    ).limit(20)
)

In [ ]:
%sql
SELECT *
FROM workspace.sanchari_silver.youtube_video_daily
LIMIT 10;